<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_16_practicum_hashing/note_lesson_16_hashing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практикум П3 — Хеш-структури: Two Sum, frequency counters, anagrams

> Третій і останній практикум алгоритмічної трійки: **П1 (позиція 8) — EVALUATE** ("наскільки дороге моє рішення?") → **П2 (позиція 11) — CHOOSE** ("яку властивість даних я можу використати?") → **П3 (тут) — REPRESENT** ("чи можу я змінити представлення даних, щоб отримати кращий алгоритм?"). Цей урок будується навколо однієї ідеї: **представлення даних визначає, які операції дешеві** — список пам'ятає порядок, але шукає лінійно; `dict`/`set` шукають майже миттєво ціною втрати порядку.

## 🔁 RETRIEVE — пригадай Уроки 6 і П2 (без підглядання)

1. Що робить `counts[char] = counts.get(char, 0) + 1`, і для чого тут саме `.get(key, 0)`, а не `counts[char]`?
2. На П2 ти розв'язував Two Sum на **відсортованих** даних `numbers = [1, 2, 4, 7, 11, 15]`, `target = 9`, двома вказівниками. Яку саме пару значень знайшов той розв'язок?
3. Чому взагалі потрібно, щоб дані були відсортовані для two-pointer підходу — що саме ламається, якщо дані невідсортовані?

<details>
<summary>Відповіді</summary>

1. Це патерн **Counting**: перебираємо елементи й накопичуємо, скільки разів зустрівся кожен. `.get(char, 0)` повертає `0`, якщо символ ще не траплявся (замість `KeyError`, який кинув би `counts[char]`), тож `+ 1` завжди коректний.
2. Значення `2` і `7` (`2 + 7 == 9`).
3. Two-pointer покладається на **монотонність**: якщо сума занадто велика — зменшуємо правий вказівник (менше значення), якщо занадто мала — збільшуємо лівий (більше значення). Ця логіка коректна, лише коли масив відсортований — інакше зменшення/збільшення вказівника не гарантує зменшення/збільшення суми.

</details>

## 📖 CONCEPT + 🛠️ CREATE — Two Sum, тепер на невідсортованих даних

Та сама задача, той самий `target = 9`, ті самі числа — але тепер у **невідсортованому** порядку:

```python
numbers = [11, 2, 15, 7, 1, 4]
target = 9
```

«Розв'яжи тим самим способом, що на П2» — спробуємо буквально той самий two-pointer підхід напряму, без сортування:

In [1]:
numbers = [11, 2, 15, 7, 1, 4]
target = 9


def two_pointer_two_sum(nums, target):
    """Той самий підхід, що на П2: два вказівники, що сходяться.
    Коректно працює лише для ВІДСОРТОВАНИХ даних."""
    left, right = 0, len(nums) - 1
    while left < right:
        s = nums[left] + nums[right]
        if s == target:
            return left, right
        elif s < target:
            left += 1
        else:
            right -= 1
    return None


result = two_pointer_two_sum(numbers, target)
print(f"two_pointer_two_sum({numbers}, {target}) -> {result}")

assert result is None, "На невідсортованих даних two-pointer не повинен знайти правильну пару"
print("Підтверджено: two-pointer з П2 не працює напряму на цих даних без сортування.")

two_pointer_two_sum([11, 2, 15, 7, 1, 4], 9) -> None
Підтверджено: two-pointer з П2 не працює напряму на цих даних без сортування.


`two_pointer_two_sum` повернув `None` — не тому що пари `9` не існує (вона є: `2 + 7`), а тому що `numbers[0] = 11` уже більше за `target`, і алгоритм весь час зменшує правий вказівник, так і не дійшовши до пари `(2, 7)`. Монотонність, на яку покладається two-pointer, тут відсутня.

**Варіант 1 — відсортувати спочатку.** Подивимось, що це коштує:

In [2]:
sorted_numbers = sorted(numbers)  # O(n log n) — сортування не безкоштовне
print(f"Відсортовано: {sorted_numbers}")

left, right = 0, len(sorted_numbers) - 1
while left < right:
    s = sorted_numbers[left] + sorted_numbers[right]
    if s == target:
        break
    elif s < target:
        left += 1
    else:
        right -= 1

print(f"Знайдена пара ЗНАЧЕНЬ у відсортованому масиві: {sorted_numbers[left]} і {sorted_numbers[right]}")

assert (sorted_numbers[left], sorted_numbers[right]) == (2, 7)

Відсортовано: [1, 2, 4, 7, 11, 15]
Знайдена пара ЗНАЧЕНЬ у відсортованому масиві: 2 і 7


Значення знайдено правильно — але це позиції `left`/`right` **у відсортованому масиві**, а не оригінальні індекси в `numbers`. Щоб повернути індекси з вихідного (невідсортованого) списку, довелося б додатково шукати `numbers.index(2)` і `numbers.index(7)` — а це знову лінійний пошук, для кожного з них. Ціна цього шляху: **сортування O(n log n) + відновлення індексів**.

**Варіант 2 — пам'ятати, що вже бачили.** Замість сортування всього масиву наперед, проходимо його один раз і для кожного числа перевіряємо: «а чи бачив я вже число, яке доповнює це до `target`?» Відповідь на це питання — саме те, для чого `dict`/`set` створені: перевірка належності майже за O(1).

### Чому `in` для `dict`/`set` майже миттєвий

Список перевіряє `x in list` **лінійно**: порівнює `x` з кожним елементом по черзі, поки не знайде збіг або не дійде до кінця — O(n). `dict`/`set` натомість обчислюють від `x` число (**хеш**) і одразу переходять до потрібного "кошика" в пам'яті — не потрібно порівнювати з усіма елементами. Це і є вся глибина, потрібна тут: `hash(x) → кошик → перевірка`, без деталей про колізії чи внутрішню реалізацію CPython. Побачимо різницю на реальних вимірах:

In [3]:
import time

big_list = list(range(50_000))
big_set = set(big_list)

# -1 свідомо відсутній — це НАЙГІРШИЙ випадок для list (повний прохід без збігу)
probe_values = [49_999, 25_000, 0, -1]


def time_membership(container, values, repeats=300):
    start = time.perf_counter()
    for _ in range(repeats):
        for v in values:
            v in container
    return time.perf_counter() - start


list_time = time_membership(big_list, probe_values)
set_time = time_membership(big_set, probe_values)

print(f"list: {list_time:.4f} с")
print(f"set:  {set_time:.4f} с")
print(f"set швидший приблизно у {list_time / max(set_time, 1e-9):.0f} разів")

assert set_time < list_time / 10, "На такому розмірі set має бути істотно швидшим за list"
print("Підтверджено: перевірка належності в set на порядки швидша, ніж у list, на великих даних.")

list: 0.2393 с
set:  0.0000 с
set швидший приблизно у 7712 разів
Підтверджено: перевірка належності в set на порядки швидша, ніж у list, на великих даних.


Різниця не випадкова й не залежить від конкретного заліза — вона структурна: `list` завжди виконує роботу, пропорційну кількості елементів (O(n)); `set`/`dict` виконують сталу (в середньому) кількість кроків незалежно від розміру (O(1) у середньому). Саме цю властивість зараз використаємо для Two Sum.

In [4]:
def two_sum_hash(nums, target):
    """Один прохід: для кожного числа перевіряємо, чи вже бачили потрібну пару."""
    seen = {}  # значення -> індекс, де воно зустрілось
    for i, num in enumerate(nums):
        complement = target - num
        if complement in seen:          # O(1) в середньому
            return seen[complement], i
        seen[num] = i
    return None


result = two_sum_hash(numbers, target)
print(f"two_sum_hash({numbers}, {target}) -> {result}")

assert result is not None
i, j = result
assert numbers[i] + numbers[j] == target
assert {i, j} == {1, 3}
print(f"Знайдено: numbers[{i}]={numbers[i]}, numbers[{j}]={numbers[j]} -> {numbers[i]} + {numbers[j]} = {target}")
print("Один прохід масиву, оригінальні індекси одразу, сортування не знадобилось.")

two_sum_hash([11, 2, 15, 7, 1, 4], 9) -> (1, 3)
Знайдено: numbers[1]=2, numbers[3]=7 -> 2 + 7 = 9
Один прохід масиву, оригінальні індекси одразу, сортування не знадобилось.


**Головний висновок цього практикуму:** алгоритм, яким варто розв'язувати задачу, залежить не тільки від самої задачі, а й від **представлення й властивостей даних**. Той самий Two Sum на відсортованих даних (П2) розв'язується двома вказівниками без жодної додаткової пам'яті; ті самі числа в невідсортованому вигляді (тут) розв'язуються ефективніше зміною представлення — накопиченням побаченого в `dict`. Це пряме застосування центральної моделі курсу: *дані → перетворення (зміна представлення) → організація логіки*.

Далі — той самий принцип на двох інших задачах: **RAW DATA → зміна представлення → dict/set → дешевші операції → інший алгоритм стає можливим.**

## 🛠️ CREATE — Частотний аналіз: не просто рахувати, а використовувати O(1)-доступ

На Уроці 6 патерн Counting уже будував `{значення: скільки_разів}`. Тепер додаємо дію, для якої важливий **дешевий доступ до вже побаченого**: перевірку на дублікати та пошук найчастішого елемента.

In [5]:
def has_duplicates(items):
    """Чи є в items хоча б одне повторення? Перевірка належності в set — O(1) в середньому."""
    seen = set()
    for item in items:
        if item in seen:
            return True
        seen.add(item)
    return False


assert has_duplicates([1, 2, 3, 2]) is True
assert has_duplicates([1, 2, 3]) is False
assert has_duplicates([]) is False
assert has_duplicates([5]) is False
assert has_duplicates([7, 7, 7]) is True
print("OK — has_duplicates: порожній список, самі унікальні, дублікат, усі однакові — усі перевірки пройдено")

OK — has_duplicates: порожній список, самі унікальні, дублікат, усі однакові — усі перевірки пройдено


In [6]:
def most_frequent(items):
    """Найчастіший елемент — Counting-патерн з Уроку 6 + max(..., key=...)."""
    if not items:
        return None
    counts = {}
    for item in items:
        counts[item] = counts.get(item, 0) + 1
    return max(counts, key=counts.get)


assert most_frequent([1, 2, 2, 3, 2]) == 2
assert most_frequent([5]) == 5
assert most_frequent([]) is None
assert most_frequent(["a", "b", "a", "c", "a", "b"]) == "a"
print("OK — most_frequent: перевірки пройдено, включно з порожнім входом")

OK — most_frequent: перевірки пройдено, включно з порожнім входом


## 🛠️ CREATE — Анаграми: порівняння через представлення, а не сортування

Два рядки — анаграми, якщо в них однакові літери з однаковою кількістю кожної (порядок не важливий). «Однакові літери з однаковою кількістю» — це буквально те, що частотний словник уже вміє порівнювати:

In [7]:
def char_counts(s):
    counts = {}
    for ch in s:
        counts[ch] = counts.get(ch, 0) + 1
    return counts


def is_anagram(a, b):
    """Регістрозалежна перевірка: 'A' і 'a' — різні символи."""
    if len(a) != len(b):
        return False
    return char_counts(a) == char_counts(b)


assert is_anagram("listen", "silent") is True
assert is_anagram("triangle", "integral") is True
assert is_anagram("apple", "papel") is True
assert is_anagram("apple", "appl") is False       # різна довжина
assert is_anagram("apple", "banana") is False      # різні частоти символів
assert is_anagram("Listen", "silent") is False     # регістрозалежно: 'L' != 'l'
print("OK — is_anagram: довжина, регістр і різні частоти перевірені окремо")

OK — is_anagram: довжина, регістр і різні частоти перевірені окремо


Альтернатива без `dict` — відсортувати обидва рядки й порівняти: `sorted(a) == sorted(b)`. Результат той самий, але сортування коштує O(n log n) проти O(n) у dict-підході — та сама ціна, що й у Варіанті 1 для Two Sum вище:

In [8]:
def is_anagram_sorted(a, b):
    return sorted(a) == sorted(b)


assert is_anagram_sorted("listen", "silent") == is_anagram("listen", "silent")
assert is_anagram_sorted("apple", "banana") == is_anagram("apple", "banana")
assert is_anagram_sorted("Listen", "silent") == is_anagram("Listen", "silent")
print("OK — sorted()-підхід дає той самий результат, але коштує O(n log n) замість O(n)")

OK — sorted()-підхід дає той самий результат, але коштує O(n log n) замість O(n)


## 🔄 TRANSFER — перший символ, що не повторюється

Напиши `first_unique_char(s)`, яка повертає **перший** символ рядка `s`, що зустрічається в ньому рівно один раз, або `None`, якщо такого немає. Той самий принцип: спочатку побудувати частотне представлення (`dict`), потім скористатись ним за O(1)-доступом на кожному символі:

In [9]:
def first_unique_char(s):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    counts = char_counts(s)
    for ch in s:
        if counts[ch] == 1:
            return ch
    return None
    # END SOLUTION


assert first_unique_char("aabbc") == "c"
assert first_unique_char("aabbcc") is None
assert first_unique_char("x") == "x"
assert first_unique_char("") is None
assert first_unique_char("swiss") == "w"
print("OK — first_unique_char: усі перевірки пройдено")

OK — first_unique_char: усі перевірки пройдено


## ✅ Самоперевірка (5 запитань)

**1.** Чому `two_pointer_two_sum(numbers, target)` повернув `None`, хоча пара `(2, 7)` існує в `numbers`?

<details><summary>Відповідь</summary>Two-pointer покладається на монотонність відсортованих даних. На невідсортованому <code>numbers</code> перше ж число (<code>11</code>) уже більше за <code>target</code>, тож алгоритм лише зменшує правий вказівник і ніколи не доходить до правильної пари.</details>

**2.** Яку саме ціну платить підхід "спочатку відсортувати" (Варіант 1) порівняно з хеш-підходом?

<details><summary>Відповідь</summary>Сортування коштує O(n log n), і після нього знайдені позиції належать відсортованому масиву — не оригінальному, тож індекси у вихідних даних довелось би шукати окремо (знову лінійний пошук).</details>

**3.** Чому `x in big_set` набагато швидший за `x in big_list` на великих даних?

<details><summary>Відповідь</summary><code>list</code> перевіряє належність лінійно — порівнює <code>x</code> з кожним елементом по черзі (O(n)). <code>set</code>/<code>dict</code> обчислюють хеш від <code>x</code> і одразу переходять у потрібний "кошик" — стала (в середньому) кількість кроків незалежно від розміру (O(1)).</details>

**4.** Чому `is_anagram("Listen", "silent")` повертає `False`?

<details><summary>Відповідь</summary>Перевірка регістрозалежна: символ <code>'L'</code> (великий) і <code>'l'</code> (малий) — різні ключі у частотному словнику, тож частотні представлення рядків не збігаються.</details>

**5.** У чому спільний принцип Two Sum, частотного аналізу й анаграм у цьому уроці?

<details><summary>Відповідь</summary>У всіх трьох: RAW DATA → зміна представлення на <code>dict</code>/<code>set</code> → операції, що раніше коштували O(n) (пошук пари, перевірка дубліката, порівняння рядків), стають дешевшими (O(1) для окремого доступу, O(n) замість O(n log n) для порівняння загалом) — представлення даних, а не сама задача, визначає, який алгоритм можливий.</details>

### Шпаргалка

```python
# Two Sum — зміна представлення замість сортування
def two_sum_hash(nums, target):
    seen = {}
    for i, num in enumerate(nums):
        complement = target - num
        if complement in seen:      # O(1) в середньому
            return seen[complement], i
        seen[num] = i
    return None

# Перевірка дубліката — O(1) в середньому на елемент
def has_duplicates(items):
    seen = set()
    for item in items:
        if item in seen:
            return True
        seen.add(item)
    return False

# Анаграми — порівняння частотних представлень, O(n), без сортування
def is_anagram(a, b):
    if len(a) != len(b):
        return False
    return char_counts(a) == char_counts(b)
```

## Підсумок трійки П1 → П2 → П3

```text
              ALGORITHMIC THINKING
                     │
          ┌──────────┼──────────┐
          ▼          ▼          ▼
       П1            П2          П3
    EVALUATE        CHOOSE     REPRESENT
   complexity      strategy    data structure
       │              │            │
       └──────────────┼────────────┘
                      ▼
             TRADE-OFF THINKING
```

- **П1 — EVALUATE**: два правильні рішення можуть коштувати по-різному (O(1)/O(n)/O(n²)) — коректність ≠ ефективність.
- **П2 — CHOOSE**: властивості даних (відсортованість, суміжність) дозволяють обрати дешевшу стратегію (linear / binary / two-pointer / sliding window) замість "перебрати все".
- **П3 — REPRESENT**: коли потрібної властивості немає (дані невідсортовані), можна **змінити саме представлення даних** (dict/set) — і отримати дешевший алгоритм, а не просто змиритись із гіршою складністю.

Ці три практикуми разом — одна навичка: перш ніж писати код, поставити собі запитання «що я знаю про ці дані, і чи можу я представити їх інакше?»